# Module 16: Interactive Cryptographic Authentication & RBAC

### What You Will Discover
By running this notebook, you will explore password hashing with work factors (`bcrypt`), constant-time string comparisons to prevent timing attacks, and minting/verifying stateless JSON Web Tokens (JWT).

**Key Question Answered:** *Why is `password_hash == user_input_hash` vulnerable to timing side-channel attacks, and how does constant-time equality resolve it?*


In [ ]:
# Step 1: Secure password hashing with bcrypt
import bcrypt

password = 'UserSecretPassphrase2026!'
salt = bcrypt.gensalt(rounds=10)
hashed = bcrypt.hashpw(password.encode('utf-8'), salt)
print(f'Hashed string: {hashed.decode("utf-8")}')


In [ ]:
# Step 2: Verifying passwords
is_valid = bcrypt.checkpw(password.encode('utf-8'), hashed)
is_wrong = bcrypt.checkpw(b'WrongPassword', hashed)
print(f'Correct password matches: {is_valid}')
print(f'Wrong password matches:   {is_wrong}')


In [ ]:
# Step 3: Constant-time comparison for API keys and tokens
import hmac

secret_key = 'sk_live_99812401824'
candidate = 'sk_live_99812401824'
print(f'Constant-time match: {hmac.compare_digest(candidate, secret_key)}')


### 🔮 Prediction Prompt
**Before running the next cell:** In a JWT token `header.payload.signature`, if an attacker tampers with the payload JSON (e.g. changes `"role": "user"` to `"role": "admin"`) without knowing the server secret, what will `jwt.decode()` raise?


In [ ]:
# Surprising Result: Cryptographic Signature Tamper Rejection
import datetime

import jwt

JWT_SECRET = 'enterprise-secret-key-42'
payload = {'sub': 'usr_100', 'role': 'viewer', 'exp': datetime.datetime.now(datetime.UTC) + datetime.timedelta(minutes=15)}
token = jwt.encode(payload, JWT_SECRET, algorithm='HS256')

# Attacker tampers with payload by modifying token string
tampered_token = token[:-5] + 'XXXXX'
try:
    jwt.decode(tampered_token, JWT_SECRET, algorithms=['HS256'])
except jwt.InvalidSignatureError as exc:
    print(f'Caught tamper attempt: {exc}')
    print('Explanation: HMAC-SHA256 signature verification caught the payload modification immediately!')


### Validating Verified Claims and Expiration
Decoded tokens return verified claims directly in memory.


In [ ]:
decoded = jwt.decode(token, JWT_SECRET, algorithms=['HS256'])
print(f'Verified Subject: {decoded["sub"]}, Role: {decoded["role"]}')


### Role-Based Access Control (RBAC) Gate
Check user role claims against route permissions.


In [ ]:
def check_permission(claims: dict, required_role: str) -> bool:
    user_role = claims.get('role')
    role_hierarchy = {'admin': 3, 'editor': 2, 'viewer': 1}
    return role_hierarchy.get(user_role, 0) >= role_hierarchy.get(required_role, 99)

print(f'Viewer can view?  {check_permission(decoded, "viewer")}')
print(f'Viewer can admin? {check_permission(decoded, "admin")}')


### 🛠️ Interactive Challenge: Prevent Algorithm Confusion (`alg: none`)
The following decoding call does not pass an explicit `algorithms` whitelist, leaving it susceptible to algorithm confusion attacks. Fix it by specifying `algorithms=['HS256']`.


In [ ]:
# TODO: FIX ME - Explicitly enforce allowed algorithms in jwt.decode
# FIX: jwt.decode(token, JWT_SECRET, algorithms=['HS256'])
claims = jwt.decode(token, JWT_SECRET, algorithms=['HS256'])
print(f'Securely decoded claims: {claims}')


### 🏁 Summary & Next Steps
- Use `bcrypt` with work factor 12 for password hashing.
- Use `hmac.compare_digest` for constant-time secret comparison.
- Always specify explicit `algorithms=['HS256']` in `jwt.decode()`.
- Run `python 01_password_hashing_demo.py` and `python 02_jwt_tokens_and_rbac_demo.py`.
- Complete [PROJECT_GUIDE.md](PROJECT_GUIDE.md) to implement the security guard.
